# 07 — NCM Scoring: Attractiveness Ranking

**Purpose:** Rank NCM-4 headings by composite attractiveness score for freight forwarding
opportunities, combining 6 weighted criteria.

**Inputs:**
- `outputs/data/mart_ncm4_annual.parquet` — volume, growth, HHI, dominant mode/URF
- `outputs/data/mart_sh6_via.parquet` — freight intensity by product × mode
- `outputs/data/mart_ncm4_urf.parquet` — URF breakdown for route scoring
- `Config/config.xlsx` sheets `scoring_weights` and `scoring_thresholds`

**Output:** Ranked top-N table + scatter plot (volume vs CAGR, bubble=HHI, color=modal match)

**Scoring criteria (weights in config.xlsx):**
1. Volume — total FOB last complete year
2. Crescimento — CAGR over N years
3. Concentração — HHI URF (low HHI = fragmented = more opportunity)
4. Modal — % air share; flag if matches target operator profile
5. Rota — % volume via target URFs (GRU/VCP/Santos)
6. Margem de frete — median FREIGHT_PCT_FOB

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "outputs" / "data"
CHART_DIR    = PROJECT_ROOT / "outputs" / "charts"
CONFIG_XLSX  = PROJECT_ROOT / "Config" / "config.xlsx"
CHART_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

## Parameters (from Config/config.xlsx)

In [ ]:
_cfg = pd.read_excel(CONFIG_XLSX, sheet_name=None)

# Scoring weights (must sum to 1)
weights = _cfg["scoring_weights"].set_index("criterio")["peso"].to_dict()

# Scoring thresholds
thresh = _cfg["scoring_thresholds"].set_index("parametro")["valor"]
TOP_N              = int(thresh["top_n_ncms"])
VOL_MIN_FOB        = float(thresh["vol_min_fob_usd"])
CAGR_YEARS         = int(thresh["crescimento_cagr_anos"])
HHI_PULVERIZADO    = float(thresh["hhi_pulverizado"])
HHI_CONCENTRADO    = float(thresh["hhi_concentrado"])
FRETE_ALTO_PCT     = float(thresh["frete_alto_pct"])

# Target URFs for route scoring (GRU=1066, VCP=1090, Santos=0817671)
# These are the most common codes for air (GRU/VCP) and sea (Santos)
TARGET_URFS = ["01066", "01090", "09710"]  # GRU, Viracopos, Santos

print(f"Weights : {weights}")
print(f"Sum     : {sum(weights.values()):.2f}")
print(f"Top N   : {TOP_N}")
print(f"Vol min : USD {VOL_MIN_FOB:,.0f}")
print(f"CAGR yrs: {CAGR_YEARS}")

## 1. Load Base Data

In [ ]:
annual_path = str(DATA_DIR / "mart_ncm4_annual.parquet").replace("\\", "/")
via_path    = str(DATA_DIR / "mart_sh6_via.parquet").replace("\\", "/")
urf_path    = str(DATA_DIR / "mart_ncm4_urf.parquet").replace("\\", "/")

con = duckdb.connect()

annual = con.execute(f"SELECT * FROM read_parquet('{annual_path}')").df()
print(f"mart_ncm4_annual : {annual.shape[0]:,} rows, years {annual['CO_ANO'].min()}–{annual['CO_ANO'].max()}")
print(f"Distinct NCM-4   : {annual['CO_POSICAO'].nunique():,}")

## 2. Determine Reference Year

In [ ]:
# Use the most recent year with data for all 12 months (complete year)
# Proxy: the annual mart already aggregates; we use max year present in data
# If the latest year is partial, we rely on the mart for volume but flag it
all_years    = sorted(annual["CO_ANO"].unique())
REF_YEAR     = all_years[-1]           # most recent year in the mart
BASE_YEAR    = REF_YEAR - CAGR_YEARS   # CAGR base

print(f"Reference year : {REF_YEAR}")
print(f"CAGR base year : {BASE_YEAR}  ({CAGR_YEARS}-year CAGR)")
print(f"All years in mart: {all_years}")

ref  = annual[annual["CO_ANO"] == REF_YEAR].copy()
base = annual[annual["CO_ANO"] == BASE_YEAR][["CO_POSICAO", "total_fob"]].rename(
    columns={"total_fob": "fob_base"}
)

print(f"\nNCM-4 with data in {REF_YEAR}: {len(ref):,}")
print(f"NCM-4 with data in {BASE_YEAR}: {len(base):,}")

## 3. Compute Scoring Criteria

In [ ]:
# ── Merge base year for CAGR ──────────────────────────────────────────────────
df = ref.merge(base, on="CO_POSICAO", how="left")

# Minimum volume filter
df = df[df["total_fob"] >= VOL_MIN_FOB].copy()
print(f"NCM-4 after volume filter (>= USD {VOL_MIN_FOB:,.0f}): {len(df):,}")

# ── Criterion 1: Volume ───────────────────────────────────────────────────────
# Log scale: higher FOB = higher score (percentile rank 0–1)
df["log_fob"] = np.log1p(df["total_fob"])
df["score_volume"] = df["log_fob"].rank(pct=True)

# ── Criterion 2: Crescimento (CAGR) ──────────────────────────────────────────
# CAGR = (fob_ref / fob_base)^(1/n) - 1; NaN where base is missing/zero
df["cagr"] = np.where(
    (df["fob_base"] > 0) & df["fob_base"].notna(),
    (df["total_fob"] / df["fob_base"]) ** (1 / CAGR_YEARS) - 1,
    np.nan,
)
# Clip extreme CAGR at 5th/95th before ranking to reduce distortion
p5, p95 = df["cagr"].quantile(0.05), df["cagr"].quantile(0.95)
df["cagr_clipped"]     = df["cagr"].clip(p5, p95)
df["score_crescimento"] = df["cagr_clipped"].rank(pct=True)

# ── Criterion 3: Concentração (HHI URF) ──────────────────────────────────────
# Low HHI = fragmented market = more opportunity → invert
df["score_concentracao"] = (1 - df["hhi_urf"].rank(pct=True))  # invert

print("\nCriteria 1–3 computed.")
df[["CO_POSICAO", "total_fob", "cagr", "hhi_urf",
    "score_volume", "score_crescimento", "score_concentracao"]].head()

In [ ]:
# ── Criterion 4: Modal (air share) ───────────────────────────────────────────
# CO_VIA = '4' = air freight in SISCOMEX VIA reference
# Use mart_sh6_via aggregated to CO_POSICAO (first 4 chars of CO_SH6)
via = con.execute(f"""
    SELECT
        LEFT(CO_SH6, 4)              AS CO_POSICAO,
        SUM(VL_FOB)                  AS fob_total,
        SUM(CASE WHEN CO_VIA = '4' THEN VL_FOB ELSE 0 END) AS fob_air
    FROM read_parquet('{via_path}')
    GROUP BY LEFT(CO_SH6, 4)
""").df()

via["air_share"] = via["fob_air"] / via["fob_total"].replace(0, np.nan)
df = df.merge(via[["CO_POSICAO", "air_share"]], on="CO_POSICAO", how="left")

# Score: higher air share = higher value per kg = better margin for air freight forwarders
df["score_modal"] = df["air_share"].rank(pct=True)

print(f"Air share data joined for {via['CO_POSICAO'].nunique():,} NCM-4 headings")

# ── Criterion 5: Rota (target URF share) ─────────────────────────────────────
target_urfs_sql = ", ".join(f"'{u}'" for u in TARGET_URFS)
urf_df = con.execute(f"""
    SELECT
        CO_POSICAO,
        SUM(VL_FOB)                                                       AS fob_total_urf,
        SUM(CASE WHEN CO_URF IN ({target_urfs_sql}) THEN VL_FOB ELSE 0 END) AS fob_target_urf
    FROM read_parquet('{urf_path}')
    GROUP BY CO_POSICAO
""").df()

urf_df["target_urf_share"] = urf_df["fob_target_urf"] / urf_df["fob_total_urf"].replace(0, np.nan)
df = df.merge(urf_df[["CO_POSICAO", "target_urf_share"]], on="CO_POSICAO", how="left")
df["score_rota"] = df["target_urf_share"].rank(pct=True)

print(f"URF data joined. Target URFs: {TARGET_URFS}")

In [ ]:
# ── Criterion 6: Margem de frete ─────────────────────────────────────────────
# median_freight_pct from mart_ncm4_annual (already computed there)
df["score_margem_frete"] = df["median_freight_pct"].rank(pct=True)

# ── Composite score ───────────────────────────────────────────────────────────
score_cols = {
    "volume"       : "score_volume",
    "crescimento"  : "score_crescimento",
    "concentracao" : "score_concentracao",
    "modal"        : "score_modal",
    "rota"         : "score_rota",
    "margem_frete" : "score_margem_frete",
}

df["score_total"] = sum(
    weights.get(criterio, 0) * df[col].fillna(0.5)  # neutral fill for missing
    for criterio, col in score_cols.items()
)

df["rank"] = df["score_total"].rank(ascending=False, method="min").astype(int)
df_ranked  = df.sort_values("score_total", ascending=False).reset_index(drop=True)

print(f"Scoring complete. Top {TOP_N} NCMs:")
print(f"  Max score : {df_ranked['score_total'].max():.3f}")
print(f"  Min score : {df_ranked['score_total'].min():.3f}")

## 4. Top-N Ranking Table

In [ ]:
from IPython.display import display

display_cols = [
    "rank", "CO_POSICAO",
    "total_fob", "cagr", "hhi_urf", "air_share", "target_urf_share", "median_freight_pct",
    "score_volume", "score_crescimento", "score_concentracao",
    "score_modal", "score_rota", "score_margem_frete",
    "score_total",
]

top_n = df_ranked.head(TOP_N)[display_cols].copy()

# Formatting helpers
top_n["total_fob"]          = top_n["total_fob"].map("${:,.0f}".format)
top_n["cagr"]               = top_n["cagr"].map(lambda x: f"{x*100:+.1f}%" if pd.notna(x) else "—")
top_n["hhi_urf"]            = top_n["hhi_urf"].map(lambda x: f"{x:,.0f}" if pd.notna(x) else "—")
top_n["air_share"]          = top_n["air_share"].map(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "—")
top_n["target_urf_share"]   = top_n["target_urf_share"].map(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "—")
top_n["median_freight_pct"] = top_n["median_freight_pct"].map(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "—")
for sc in ["score_volume","score_crescimento","score_concentracao",
           "score_modal","score_rota","score_margem_frete","score_total"]:
    top_n[sc] = top_n[sc].map("{:.3f}".format)

top_n.columns = [
    "#", "NCM-4",
    f"FOB {REF_YEAR}", f"CAGR {CAGR_YEARS}a", "HHI-URF", "Air%", "TargetURF%", "Frete%FOB",
    "S.Vol", "S.Cresc", "S.Conc", "S.Modal", "S.Rota", "S.Frete",
    "SCORE",
]

display(top_n.style.hide(axis="index"))

## 5. Scatter Plot: Volume × Growth × HHI × Modal

In [ ]:
plot_df = df_ranked.head(TOP_N * 2).copy()  # slightly wider pool for the chart
plot_df = plot_df.dropna(subset=["cagr", "total_fob"])

# Bubble size: inverse HHI (more fragmented = bigger bubble)
max_hhi  = plot_df["hhi_urf"].fillna(HHI_CONCENTRADO).clip(upper=10000)
bub_size = (10000 - max_hhi.fillna(10000)) / 80 + 20  # range ~20–145

# Color: high air share = orange, else blue
AIR_THRESHOLD = 0.30
colors = [
    "#E07B39" if (not pd.isna(a) and a >= AIR_THRESHOLD) else "#4A7FB5"
    for a in plot_df["air_share"]
]

fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(
    np.log10(plot_df["total_fob"]),
    plot_df["cagr"] * 100,
    s=bub_size,
    c=colors,
    alpha=0.7,
    edgecolors="white",
    linewidths=0.5,
)

# Label top N by score
for _, row in df_ranked.head(TOP_N).iterrows():
    if pd.isna(row["cagr"]) or pd.isna(row["total_fob"]):
        continue
    ax.annotate(
        row["CO_POSICAO"],
        (np.log10(row["total_fob"]), row["cagr"] * 100),
        fontsize=6.5,
        textcoords="offset points",
        xytext=(4, 3),
        color="#333333",
    )

ax.axhline(0, color="#aaaaaa", linewidth=0.8, linestyle="--")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"$10^{{{x:.0f}}}$"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:+.0f}%"))

# Legend proxies
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#E07B39', markersize=9,
           label=f'Air share ≥ {AIR_THRESHOLD*100:.0f}%'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#4A7FB5', markersize=9,
           label=f'Air share < {AIR_THRESHOLD*100:.0f}%'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#888888', markersize=5,
           label='Bubble size ∝ market fragmentation (1/HHI)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc="lower right")

ax.set_xlabel(f"FOB {REF_YEAR} (log scale, USD)", fontsize=10)
ax.set_ylabel(f"{CAGR_YEARS}-year CAGR", fontsize=10)
ax.set_title(f"NCM-4 Attractiveness — Volume × Growth × HHI × Modal ({REF_YEAR})", fontsize=12)

plt.tight_layout()
out = CHART_DIR / "07_ncm_scoring_scatter.png"
fig.savefig(out, bbox_inches="tight")
plt.show()
print(f"Chart saved → {out}")

## 6. Score Component Breakdown (top-10 heatmap)

In [ ]:
score_raw_cols = [
    "score_volume", "score_crescimento", "score_concentracao",
    "score_modal", "score_rota", "score_margem_frete",
]
heat = df_ranked.head(10).set_index("CO_POSICAO")[score_raw_cols].astype(float)
heat.columns = ["Volume", "Crescimento", "Concentração", "Modal", "Rota", "Margem Frete"]

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(heat.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, fontsize=9)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=9)

for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        val = heat.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="black" if 0.25 < val < 0.75 else "white", fontsize=8)

plt.colorbar(im, ax=ax, label="Percentile score (0–1)")
ax.set_title("Score Components — Top 10 NCM-4", fontsize=11)
plt.tight_layout()
out2 = CHART_DIR / "07_ncm_scoring_heatmap.png"
fig.savefig(out2, bbox_inches="tight")
plt.show()
print(f"Chart saved → {out2}")

## 7. Strategic Read-out

In [ ]:
top10 = df_ranked.head(10)

high_air   = df_ranked[df_ranked["air_share"] >= AIR_THRESHOLD].head(5)["CO_POSICAO"].tolist()
high_frete = df_ranked[df_ranked["median_freight_pct"] >= FRETE_ALTO_PCT].head(5)["CO_POSICAO"].tolist()
low_hhi    = df_ranked[df_ranked["hhi_urf"] < HHI_PULVERIZADO].head(5)["CO_POSICAO"].tolist()

print("=" * 60)
print(f"  NCM SCORING — {REF_YEAR}  (top {TOP_N} NCMs)")
print("=" * 60)
print(f"\nTop 10 NCM-4 by composite score:")
for _, row in top10.iterrows():
    cagr_str  = f"{row['cagr']*100:+.1f}%" if pd.notna(row['cagr']) else "N/A"
    air_str   = f"{row['air_share']*100:.0f}%" if pd.notna(row['air_share']) else "N/A"
    print(f"  {int(row['rank']):>3}. {row['CO_POSICAO']}  "
          f"score={row['score_total']:.3f}  "
          f"FOB=${row['total_fob']/1e6:.1f}M  "
          f"CAGR={cagr_str}  "
          f"Air={air_str}")

print(f"\nHigh air share (≥{AIR_THRESHOLD*100:.0f}%) in top pool : {high_air}")
print(f"High freight margin (≥{FRETE_ALTO_PCT*100:.0f}%) in top pool : {high_frete}")
print(f"Fragmented markets (HHI < {HHI_PULVERIZADO:.0f}) in top pool : {low_hhi}")
print(f"\nWeights applied: {weights}")
print("\n→ Adjust weights or thresholds in Config/config.xlsx and re-run.")